In [1]:
from datetime import datetime
from pathlib import Path
import re
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

## Anomali Usaha

In [13]:
folder = Path("../SE2026 anomali-bagi prov/")
files = list(folder.glob("Data_Mikro_Anomali_usaha*.xlsx"))

if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder scrap_anomali_usaha")


print(f"Ditemukan {len(files)} file:")
for f in files:
    print(" -", f.name)


def dedup_columns(cols):
    cols = list(cols)
    seen = {}
    result = []
    for c in cols:
        c = str(c) if pd.notna(c) else "kolom_kosong"
        if c in seen:
            seen[c] += 1
            result.append(f"{c}_{seen[c]}")
        else:
            seen[c] = 0
            result.append(c)
    return result


list_df = []
for f in files:
    raw = pd.read_excel(f, header=None, dtype=str)

    header = raw.iloc[3]

    # Cari posisi kolom "Cheklist (1/0)", potong sampai kolom itu saja
    idx_batas = header[header.astype(str).str.strip() == "Cheklist (1/0)"].index
    if len(idx_batas) == 0:
        raise ValueError(f"Kolom 'Cheklist (1/0)' tidak ditemukan di file {f.name}")
    batas = idx_batas[0]

    header = header.iloc[: batas + 1]  # ambil header sampai kolom itu saja
    data = raw.iloc[5:, : batas + 1].copy()  # ambil data sampai kolom itu saja
    data.columns = dedup_columns(header)

    data["source_file"] = f.name
    list_df.append(data)

df_gabungan = pd.concat(list_df, ignore_index=True)
print(f"\nTotal baris setelah digabung: {len(df_gabungan)}")
df_gabungan.head()

Ditemukan 17 file:
 - Data_Mikro_Anomali_usaha_1601_20260713_042514.xlsx
 - Data_Mikro_Anomali_usaha_1602_20260713_042540.xlsx
 - Data_Mikro_Anomali_usaha_1603_20260713_042558.xlsx
 - Data_Mikro_Anomali_usaha_1604_20260713_042617.xlsx
 - Data_Mikro_Anomali_usaha_1605_20260713_042633.xlsx
 - Data_Mikro_Anomali_usaha_1606_20260713_042655.xlsx
 - Data_Mikro_Anomali_usaha_1607_20260713_042714.xlsx
 - Data_Mikro_Anomali_usaha_1608_20260713_042734.xlsx
 - Data_Mikro_Anomali_usaha_1609_20260713_042752.xlsx
 - Data_Mikro_Anomali_usaha_1610_20260713_042810.xlsx
 - Data_Mikro_Anomali_usaha_1611_20260713_042830.xlsx
 - Data_Mikro_Anomali_usaha_1612_20260713_042846.xlsx
 - Data_Mikro_Anomali_usaha_1613_20260713_042906.xlsx
 - Data_Mikro_Anomali_usaha_1671_20260713_042925.xlsx
 - Data_Mikro_Anomali_usaha_1672_20260713_042940.xlsx
 - Data_Mikro_Anomali_usaha_1673_20260713_043000.xlsx
 - Data_Mikro_Anomali_usaha_1674_20260713_043017.xlsx

Total baris setelah digabung: 11852


,No,Nama Usaha,Kode Prov,Nama Provinsi,Kode Kab/Kota,Nama Kab/Kota,Kode Kec,Nama Kecamatan,Kode Desa,Nama Desa/Kel,...,Nama Anomali,Tindak Lanjut,ID Petugas,Email Petugas,Link Fasih,Produk Sendiri Value,Rasio,Petugas anomali,Cheklist (1/0),source_file
0,1,SOSIS GORENG (ADEL),16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601090,PENINJAUAN,1601090034,SAUNG NAGA,...,Jumlah Anomali Data 1 (Biaya Produksi Dominan)...,Belum Ditindaklanjuti,-,-,https://fasih-sm.bps.go.id/app/assignment-deta...,2,64.3564,nuhar@bps.go.id,0,Data_Mikro_Anomali_usaha_1601_20260713_042514....
1,2,WARUNG SEMBAKO,16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601052,LENGKITI,1601052017,WAY HELING,...,Jumlah Anomali Data 1 (Biaya Produksi Dominan)...,Belum Ditindaklanjuti,-,-,https://fasih-sm.bps.go.id/app/assignment-deta...,2,53.0303,nuhar@bps.go.id,0,Data_Mikro_Anomali_usaha_1601_20260713_042514....
2,3,OJEK KELILING(ERDANI YUHARSA),16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601130,BATU RAJA TIMUR,1601130007,KEMALA RAJA,...,Jumlah Anomali Data 1 (Biaya Produksi Dominan)...,Belum Ditindaklanjuti,-,-,https://fasih-sm.bps.go.id/app/assignment-deta...,2,85.7143,nuhar@bps.go.id,0,Data_Mikro_Anomali_usaha_1601_20260713_042514....
3,4,KUE JAJANAN PASAR RUMIANAH,16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601130,BATU RAJA TIMUR,1601130007,KEMALA RAJA,...,Jumlah Anomali Data 1 (Biaya Produksi Dominan)...,Belum Ditindaklanjuti,-,-,https://fasih-sm.bps.go.id/app/assignment-deta...,2,85.4701,nuhar@bps.go.id,0,Data_Mikro_Anomali_usaha_1601_20260713_042514....
4,5,WARUNG NEK KOKO,16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601082,ULU OGAN,1601082004,GUNUNG TIGA,...,Jumlah Anomali Data 1 (Biaya Produksi Dominan)...,Belum Ditindaklanjuti,-,-,https://fasih-sm.bps.go.id/app/assignment-deta...,2,100,nuhar@bps.go.id,0,Data_Mikro_Anomali_usaha_1601_20260713_042514....


In [16]:
from datetime import datetime
df_gabungan = df_gabungan.drop(columns=["source_file"])
tanggal_jam = datetime.now().strftime("%Y%m%d_%H%M%S")
df_gabungan.to_excel(
    folder / f"anomali_usaha_1_prov_sharing{tanggal_jam}.xlsx", index=False
)

## Missing NIK

In [17]:
folder = Path("../SE2026 anomali-bagi prov/")
files = list(folder.glob("Data_Mikro_Missing_Value_*.xlsx"))

if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder scrap_anomali_usaha")


print(f"Ditemukan {len(files)} file:")
for f in files:
    print(" -", f.name)


def dedup_columns(cols):
    cols = list(cols)
    seen = {}
    result = []
    for c in cols:
        c = str(c) if pd.notna(c) else "kolom_kosong"
        if c in seen:
            seen[c] += 1
            result.append(f"{c}_{seen[c]}")
        else:
            seen[c] = 0
            result.append(c)
    return result


list_df = []
for f in files:
    raw = pd.read_excel(f, header=None, dtype=str)

    header = raw.iloc[3]

    # Cari posisi kolom "Cheklist (1/0)", potong sampai kolom itu saja
    idx_batas = header[header.astype(str).str.strip() == "Cheklist (1/0)"].index
    if len(idx_batas) == 0:
        raise ValueError(f"Kolom 'Cheklist (1/0)' tidak ditemukan di file {f.name}")
    batas = idx_batas[0]

    header = header.iloc[: batas + 1]  # ambil header sampai kolom itu saja
    data = raw.iloc[5:, : batas + 1].copy()  # ambil data sampai kolom itu saja
    data.columns = dedup_columns(header)

    data["source_file"] = f.name
    list_df.append(data)

df_gabungan = pd.concat(list_df, ignore_index=True)
print(f"\nTotal baris setelah digabung: {len(df_gabungan)}")
df_gabungan.head()

Ditemukan 17 file:
 - Data_Mikro_Missing_Value_R7_1601_20260713031046.xlsx
 - Data_Mikro_Missing_Value_R7_1602_20260713031059.xlsx
 - Data_Mikro_Missing_Value_R7_1603_20260713031108.xlsx
 - Data_Mikro_Missing_Value_R7_1604_20260713031116.xlsx
 - Data_Mikro_Missing_Value_R7_1605_20260713031125.xlsx
 - Data_Mikro_Missing_Value_R7_1606_20260713031134.xlsx
 - Data_Mikro_Missing_Value_R7_1607_20260713031142.xlsx
 - Data_Mikro_Missing_Value_R7_1608_20260713031156.xlsx
 - Data_Mikro_Missing_Value_R7_1609_20260713031209.xlsx
 - Data_Mikro_Missing_Value_R7_1610_20260713031223.xlsx
 - Data_Mikro_Missing_Value_R7_1611_20260713031241.xlsx
 - Data_Mikro_Missing_Value_R7_1612_20260713031252.xlsx
 - Data_Mikro_Missing_Value_R7_1613_20260713031304.xlsx
 - Data_Mikro_Missing_Value_R7_1671_20260713031316.xlsx
 - Data_Mikro_Missing_Value_R7_1672_20260713031327.xlsx
 - Data_Mikro_Missing_Value_R7_1673_20260713031338.xlsx
 - Data_Mikro_Missing_Value_R7_1674_20260713031347.xlsx

Total baris setelah digabung

,No,Nama Kepala Keluarga,Kode Prov,Nama Provinsi,Kode Kab/Kota,Nama Kab/Kota,Kode Kec,Nama Kecamatan,Kode Desa,Nama Desa/Kel,Kode SLS,Sub SLS,Assignment ID,Variabel Missing,ID Petugas,Email Petugas,Link Fasih,Petugas mv,Cheklist (1/0),source_file
0,1,Rapandra,16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601081,SEMIDANG AJI,1601081005,SUKA MERINDU,0001,00,008de8d8-f9ad-45d5-b6dd-cc21872a95ec,Missing Values NIK Anggota Keluarga,af418f65-c45a-40be-93fc-d47c84c458b2,dhona268@gmail.com,https://fasih-sm.bps.go.id/app/assignment-deta...,afriandi@bps.go.id,0,Data_Mikro_Missing_Value_R7_1601_2026071303104...
1,2,SAKDIAH,16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601093,KEDATON PENINJAUAN RAYA,1601093005,SUKA PINDAH,0002,00,0b35377c-b463-4f21-b160-6a0ae1eb8583,Missing Values NIK Anggota Keluarga,bf7948aa-1150-453b-ab36-a3466efea9f2,bangben807@gmail.com,https://fasih-sm.bps.go.id/app/assignment-deta...,afriandi@bps.go.id,0,Data_Mikro_Missing_Value_R7_1601_2026071303104...
2,3,RUDIANTO,16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601090,PENINJAUAN,1601090029,PANJI JAYA,0009,00,0d2329d2-fb71-496f-84fc-2c0135046a72,Missing Values NIK Anggota Keluarga,30f09340-94cc-411b-a300-19d1db52c95c,helenelisa362@gmail.com,https://fasih-sm.bps.go.id/app/assignment-deta...,afriandi@bps.go.id,0,Data_Mikro_Missing_Value_R7_1601_2026071303104...
3,4,Davinra saputra,16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601082,ULU OGAN,1601082001,KELUMPANG,0011,00,0ece0d58-3a93-4997-bab8-0204b6c433b0,Missing Values NIK Anggota Keluarga,adcc5ce7-580a-4240-8207-3020e6fc0de2,surindayolap@gmail.com,https://fasih-sm.bps.go.id/app/assignment-deta...,afriandi@bps.go.id,0,Data_Mikro_Missing_Value_R7_1601_2026071303104...
4,5,ZUNRI,16,SUMATERA SELATAN,1601,OGAN KOMERING ULU,1601082,ULU OGAN,1601082001,KELUMPANG,0012,00,0fa3d13a-9b64-4e3e-ae38-b6c18aadd0ec,Missing Values NIK Anggota Keluarga,adcc5ce7-580a-4240-8207-3020e6fc0de2,surindayolap@gmail.com,https://fasih-sm.bps.go.id/app/assignment-deta...,afriandi@bps.go.id,0,Data_Mikro_Missing_Value_R7_1601_2026071303104...


In [18]:
from datetime import datetime

df_gabungan = df_gabungan.drop(columns=["source_file"])

tanggal_jam = datetime.now().strftime("%Y%m%d_%H%M%S")
df_gabungan.to_excel(
    folder / f"missing_nik_prov_sharing_{tanggal_jam}.xlsx", index=False
)